# Shared encoder + semantic anchors
Contains: `SharedEncoder`, anchor texts and `AnchorBank` (cosine `Y_cos`), and the co-occurrence adjacency `A_corr`.

**Input:** `train_labels` [N, 28] (multi-hot, TRAIN split) to build `A_corr`.
**Output:** the `encoder` object (one instance, never a second copy), `anchor_bank`, `adjacency`.

In [ ]:
# ===== IMPORTS =====
import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from transformers import logging as hf_logging

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
hf_logging.set_verbosity_error()

In [ ]:
GOEMOTIONS_LABELS: List[str] = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NUM_CLASSES: int = len(GOEMOTIONS_LABELS)  # 28
NEUTRAL_IDX: int = GOEMOTIONS_LABELS.index("neutral")

GOEMOTIONS_DEFINITIONS: Dict[str, str] = {
    'admiration': (
        "feeling profound respect, wonder, and high esteem toward a person for exceptional talent, skill, courage, "
        "character, or achievement; typically expressed as praise or awe, such as 'incredible work', 'you are amazing', 'what a legend'."
    ),
    'amusement': (
        "finding something funny and enjoying it: laughter, chuckling, playful wit, jokes, sarcasm taken lightly, "
        "and lighthearted entertainment; typically expressed as 'haha', 'lol', 'that is hilarious', or a witty remark."
    ),
    'anger': (
        "intense rage, fury, hostility, and outrage directed at a person or group that has wronged, insulted, "
        "or provoked the speaker; strong, heated, often aggressive wording such as 'I hate you', 'how dare they', 'this is infuriating'."
    ),
    'annoyance': (
        "mild to moderate irritation, impatience, and exasperation caused by small, repeated, or petty disturbances, "
        "inconveniences, or other people's behavior; low-intensity grumbling such as 'ugh, not again', 'this is so irritating', 'stop doing that'."
    ),
    'approval': (
        "expressing agreement, endorsement, and positive judgment of an idea, decision, opinion, or action; "
        "saying that something is right, fair, or well done, such as 'that makes sense', 'good call', 'I agree', 'fair enough', 'that is the right choice'."
    ),
    'caring': (
        "showing warmth, concern, and kindness toward someone's well-being: offering help, comfort, support, or protection, "
        "and checking on others; typically expressed as 'take care of yourself', 'I am here for you', 'hope you feel better'."
    ),
    'confusion': (
        "not understanding something that is unclear, contradictory, or illogical: puzzlement, being lost, and needing clarification; "
        "typically expressed as 'I don't get it', 'what does that mean?', 'wait, what?', 'this makes no sense to me'."
    ),
    'curiosity': (
        "an eager wish to learn more: asking questions, wanting to explore, investigate, or find out hidden or missing information; "
        "typically expressed as 'I wonder why', 'how does that work?', 'tell me more', 'I would like to know'."
    ),
    'desire': (
        "a strong wish, craving, or longing to have, do, or become something specific; wanting an object, an experience, "
        "an outcome, or a person; typically expressed as 'I really want', 'I wish I could', 'I need this', 'I long for'."
    ),
    'disappointment': (
        "feeling let down and discouraged because an expectation, hope, or promise was not met and the outcome turned out worse than hoped; "
        "a quiet dejection about something that fell short, such as 'I expected more', 'what a letdown', 'that was underwhelming', 'I hoped it would be better'."
    ),
    'disapproval': (
        "a negative judgment that something is wrong, unacceptable, or a bad idea: objecting to or condemning an action, decision, opinion, or standard, "
        "without necessarily being angry; typically expressed as 'that is not okay', 'you should not do that', 'I don't agree', 'that is a terrible idea'."
    ),
    'disgust': (
        "strong revulsion and distaste toward something repulsive, filthy, rotten, vile, or morally sickening; "
        "a physical or moral urge to recoil, typically expressed as 'that is gross', 'ew', 'disgusting', 'it makes me sick'."
    ),
    'embarrassment': (
        "feeling self-conscious, awkward, or ashamed in front of others after a blunder, slip, or exposure: blushing, wanting to hide, "
        "cringing at a public mistake; typically expressed as 'how awkward', 'I want to disappear', 'that was so embarrassing', 'I feel so silly'."
    ),
    'excitement': (
        "high-energy enthusiasm and eager anticipation about something thrilling that is happening or about to happen; "
        "can't-wait feelings, thrill, and exhilaration, typically expressed as 'I can't wait', 'so pumped', 'this is going to be awesome!', with exclamation marks."
    ),
    'fear': (
        "feeling frightened or threatened by danger, harm, or something terrifying: terror, panic, and the urge to flee or protect oneself; "
        "typically expressed as 'I am scared', 'this is terrifying', 'I'm afraid something will happen', 'run!'."
    ),
    'gratitude': (
        "feeling and expressing thankfulness and appreciation toward someone for a favor, gift, help, kindness, or support; "
        "typically expressed as 'thank you so much', 'I really appreciate it', 'thanks for your help', 'I am grateful for everything'."
    ),
    'grief': (
        "deep, heavy mourning and heartbreak after the death of a loved one or another devastating, irreversible loss; "
        "talk of someone passing away, funerals, missing the deceased, such as 'I lost my mother', 'rest in peace', 'she is gone and I can't cope'."
    ),
    'joy': (
        "a bright feeling of happiness, delight, and contentment about one's own good situation or good news in the moment; "
        "cheerful, glowing wellbeing, typically expressed as 'I am so happy', 'this made my day', 'what a wonderful feeling', 'I'm thrilled and smiling'."
    ),
    'love': (
        "deep affection, devotion, and tenderness toward a person, partner, family member, pet, or beloved thing; "
        "romantic or heartfelt attachment, typically expressed as 'I love you', 'you mean everything to me', 'my heart belongs to you', 'adore'."
    ),
    'nervousness': (
        "anxious, jittery tension and worry about an upcoming uncertain event such as an exam, interview, performance, or result; "
        "butterflies in the stomach and stage fright, typically expressed as 'I'm so nervous', 'what if it goes wrong', 'my hands are shaking'."
    ),
    'optimism': (
        "a hopeful, confident outlook that things will turn out well in the future, looking on the bright side despite difficulties; "
        "typically expressed as 'it will get better', 'I'm sure it will work out', 'things are looking up', 'better days are ahead'."
    ),
    'pride': (
        "satisfaction and dignity felt about one's own accomplishment, or about the success of one's child, family, team, or country; "
        "typically expressed as 'I did it', 'so proud of my son', 'I worked hard for this and it paid off', 'proud to be part of this'."
    ),
    'realization': (
        "the moment of suddenly understanding or noticing something that was not clear before: an insight, discovery, or the dawning of a fact, "
        "mistake, or truth; typically expressed as 'oh, now I get it', 'I just realized', 'it hit me that', 'so that's why', 'I never noticed before'."
    ),
    'relief': (
        "the feeling of tension and worry draining away because something bad did not happen or a stressful situation has finally ended well; "
        "typically expressed as 'what a relief', 'thank goodness', 'I can finally breathe', 'phew, it's over', 'glad that's behind me'."
    ),
    'remorse': (
        "guilt, regret, and sorrow about one's own wrongdoing: feeling bad for having hurt or failed someone and wishing to undo or apologize for it; "
        "typically expressed as 'I'm so sorry', 'I feel terrible about what I did', 'it's my fault', 'I should never have said that'."
    ),
    'sadness': (
        "feeling sorrowful, down, lonely, or hopeless because of hardship, rejection, or a painful situation; low mood, crying, and melancholy, "
        "typically expressed as 'I feel so sad', 'this hurts', 'I'm depressed', 'it breaks my heart', 'nothing seems to help'."
    ),
    'surprise': (
        "astonishment at something sudden, unexpected, or hard to believe: being startled or caught off guard, whether pleasantly or not; "
        "typically expressed as 'wow', 'I did not see that coming', 'no way!', 'what?!', 'I can't believe it'."
    ),
    'neutral': (
        "plain, factual, matter-of-fact language that states information, gives a description, asks a routine question, or reports an event "
        "without any emotional attitude, opinion, or feeling; objective statements such as 'the meeting is at 3 pm', 'here is the link', 'it is located downtown'."
    ),
}

assert set(GOEMOTIONS_LABELS) == set(GOEMOTIONS_DEFINITIONS), "labels and definitions must match 1-to-1"


assert set(GOEMOTIONS_LABELS) == set(GOEMOTIONS_DEFINITIONS), "labels and definitions must match 1-to-1"

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def banner(title: str) -> None:
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

def resolve_held_out(names: Sequence[str]) -> List[int]:
    """Map held-out class names to column indices (validates names, rejects 'all classes')."""
    unknown = [n for n in names if n not in GOEMOTIONS_LABELS]
    assert not unknown, f"unknown held-out classes: {unknown}"
    idx = sorted({GOEMOTIONS_LABELS.index(n) for n in names})
    assert len(idx) < NUM_CLASSES, "at least one class must remain seen"
    return idx

def make_seen_mask(num_classes: int, held_out_classes: Optional[Sequence[int]] = None) -> torch.Tensor:
    """Float mask [C]: 1.0 for seen classes, 0.0 for held-out (zero-shot) classes."""
    mask = torch.ones(num_classes)
    if held_out_classes:
        mask[torch.as_tensor(list(held_out_classes), dtype=torch.long)] = 0.0
    return mask

def _in_notebook() -> bool:
    try:
        ip = get_ipython()  # type: ignore[name-defined]  # noqa: F821
    except NameError:
        return False
    return ip is not None and getattr(ip, "kernel", None) is not None

def _show_table(df: pd.DataFrame, title: Optional[str] = None, styler=None) -> None:
    """Rich HTML table inside a notebook, plain text elsewhere."""
    if title:
        print(title)
    if _in_notebook():
        from IPython.display import display
        display(styler if styler is not None else df)
    else:
        print(df.to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
@dataclass
class Config:
    model_name: str = "roberta-base"         # HF hub id or local directory (tokenizer + weights)
    max_len: int = 64                        # max tokens per anchor definition
    seed: int = 42
    held_out_classes: List[str] = field(default_factory=list)  # names of unseen emotions, e.g. grief pride relief
    self_loop_weight: float = 1.0            # A_corr
    min_edge_prob: float = 0.0               # A_corr: drop conditional co-occurrence edges P(j|i) below this value (0 = keep all)


cfg = Config()
set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
banner("CONFIGURATION")
print(f"[ENV] device={device} | model={cfg.model_name} | seed={cfg.seed}")
print(f"[CFG] max_len={cfg.max_len} | held-out={cfg.held_out_classes or 'none (fully supervised)'}")
print(f"[TAXONOMY] {NUM_CLASSES} classes | {len(GOEMOTIONS_DEFINITIONS)} anchor definitions")

In [ ]:
## --- 7.0 Offline anchor generation ---
def build_anchor_texts(label_names: Sequence[str] = GOEMOTIONS_LABELS,
                       definitions: Dict[str, str] = GOEMOTIONS_DEFINITIONS) -> List[str]:
    """28 prompts formatted strictly as "{name}: {definition}". Fails loudly instead of silently falling back."""
    texts = []
    for name in label_names:
        assert name in definitions and definitions[name].strip(), f"missing definition for {name!r}"
        texts.append(f"{name}: {definitions[name].strip()}")
    assert len(texts) == NUM_CLASSES
    return texts


class AnchorBank(nn.Module):
    """
    Fixed semantic anchors A in R^{C x D}: encoded ONCE (offline) by the shared encoder under eval() and no_grad,
    attention-mask-weighted mean pooled, L2-normalised and stored as a registered, non-trainable buffer
    (moves with .to(device), is saved in state_dict, has requires_grad=False).
    """

    def __init__(self, anchors: torch.Tensor):
        super().__init__()
        assert anchors.dim() == 2, f"anchors must be [C, D], got {tuple(anchors.shape)}"
        self.register_buffer("A", F.normalize(anchors.detach().float(), p=2, dim=-1))

    @staticmethod
    @torch.no_grad()
    def _encode(encoder: nn.Module, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        was_training = encoder.training
        encoder.eval()  # dropout OFF -> deterministic anchors
        try:
            raw = encoder(input_ids, attention_mask)  # [C, D] mean-pooled
        finally:
            encoder.train(was_training)
        return F.normalize(raw.detach().float(), p=2, dim=-1)

    @classmethod
    def build(cls, encoder: nn.Module, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> "AnchorBank":
        bank = cls(cls._encode(encoder, input_ids, attention_mask))
        bank.register_buffer("_ids", input_ids.detach().clone(), persistent=False)   # kept for optional refresh
        bank.register_buffer("_mask", attention_mask.detach().clone(), persistent=False)
        return bank

    @classmethod
    def from_texts(cls, encoder: nn.Module, tokenizer, texts: Sequence[str], max_length: int = 64) -> "AnchorBank":
        enc = tokenizer(list(texts), padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        dev = next(encoder.parameters()).device
        return cls.build(encoder, enc["input_ids"].to(dev), enc["attention_mask"].to(dev))

    @torch.no_grad()
    def refresh_(self, encoder: nn.Module) -> None:
        """Optional: re-encode anchors with the current fine-tuned encoder (still gradient-free)."""
        self.A.copy_(self._encode(encoder, self._ids, self._mask).to(self.A.device))

    def cosine(self, h: torch.Tensor) -> torch.Tensor:
        """Base cosine similarity Y_cos = cos(h, A): [B, D] x [C, D] -> [B, C]."""
        return F.normalize(h, p=2, dim=-1) @ self.A.t()


## --- 8.0 Model architecture (modules and container) ---
class SharedEncoder(nn.Module):
    """RoBERTa backbone + attention-mask-weighted mean pooling: (input_ids, attention_mask) -> h [B, D]."""

    def __init__(self, model_name: str = "roberta-base"):
        super().__init__()
        try:
            self.backbone = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        except TypeError:  # architectures without a pooling-layer switch
            self.backbone = AutoModel.from_pretrained(model_name)
        self.hidden_size = int(self.backbone.config.hidden_size)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        tokens = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # [B, L, D]
        m = attention_mask.unsqueeze(-1).type_as(tokens)
        return (tokens * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)                                  # [B, D]

In [ ]:
# ---------------------------------------------------------------------------
# A. Leak-free adjacency matrix A_corr [C, C]
# ---------------------------------------------------------------------------
def build_adjacency_matrix(train_labels: torch.Tensor, num_classes: int = NUM_CLASSES,
                           held_out_classes: Optional[Sequence[int]] = None,
                           self_loop_weight: float = 1.0, min_edge_prob: float = 0.0) -> torch.Tensor:
    """
    Row-normalised label co-occurrence graph computed from the TRAIN split only.
      P[i, j]  = count(i and j) / count(i)          conditional co-occurrence (diagonal removed)
      held-out classes: rows AND columns are zeroed (no information flows to or from unseen classes)
      A        = P + self_loop_weight * I           identity self-loops (seen classes only)
      A[i, :] /= sum_j A[i, j]                      rows of seen classes sum to 1.0, held-out rows are all-zero
    """
    assert train_labels.dim() == 2 and train_labels.size(1) == num_classes
    held = list(held_out_classes or [])
    Y = train_labels.float()
    co = Y.t() @ Y                                              # [C, C] co-occurrence counts
    counts = Y.sum(dim=0).clamp(min=1e-9)                       # [C]
    P = co / counts.unsqueeze(1)                                # P(j | i)
    P.fill_diagonal_(0.0)
    if min_edge_prob > 0:
        P = torch.where(P >= min_edge_prob, P, torch.zeros_like(P))
    A = P + self_loop_weight * torch.eye(num_classes)
    if held:
        idx = torch.as_tensor(held, dtype=torch.long)
        A[idx, :] = 0.0
        A[:, idx] = 0.0
    A = A / A.sum(dim=1, keepdim=True).clamp(min=1e-9)
    seen = make_seen_mask(num_classes, held).bool()
    assert torch.allclose(A[seen].sum(dim=1), torch.ones(int(seen.sum())), atol=1e-5), "seen rows must sum to 1"
    if held:
        assert A[~seen].abs().sum() == 0 and A[:, ~seen].abs().sum() == 0, "held-out rows/cols must be zero"
    return A

In [ ]:
# ---------------- run: encode the 28 definitions ONCE, fixed non-trainable buffer ----------------
banner("CELL 7: OFFLINE SEMANTIC ANCHORS")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
held_out = resolve_held_out(cfg.held_out_classes)
encoder = SharedEncoder(cfg.model_name).to(device)
anchor_texts = build_anchor_texts()
anchor_bank = AnchorBank.from_texts(encoder, tokenizer, anchor_texts, max_length=cfg.max_len).to(device)

A = anchor_bank.A
norms = A.norm(dim=1)
print(f"[ANCHORS] A {tuple(A.shape)} | registered buffer: {'A' in dict(anchor_bank.named_buffers())} "
      f"| requires_grad={A.requires_grad} | row L2 norm min/max {norms.min():.4f}/{norms.max():.4f}")
print(f"[ANCHORS] example prompt: '{anchor_texts[0][:90]}'")
sim = (A @ A.t()).cpu()
sim.fill_diagonal_(-1.0)
nearest = sim.argmax(dim=1)
anchor_table = pd.DataFrame({
    "class": GOEMOTIONS_LABELS,
    "type": ["held-out" if i in set(held_out) else "seen" for i in range(NUM_CLASSES)],
    "anchor prompt": [t if len(t) <= 78 else t[:75] + "..." for t in anchor_texts],
    "nearest anchor": [GOEMOTIONS_LABELS[j] for j in nearest.tolist()],
    "cosine": sim.max(dim=1).values.numpy(),
})
_show_table(anchor_table, "\nAnchor bank summary:")

In [ ]:
# ---------------- run: A_corr from the TRAIN split only (needs `train_labels`: [N, 28] multi-hot, TRAIN split) ----------------
banner("A_corr: LABEL CO-OCCURRENCE ADJACENCY (TRAIN ONLY)")
held_out = resolve_held_out(cfg.held_out_classes)
adjacency = build_adjacency_matrix(torch.as_tensor(train_labels), NUM_CLASSES, held_out,
                                   cfg.self_loop_weight, cfg.min_edge_prob)
seen = make_seen_mask(NUM_CLASSES, held_out).bool()
print(f"[ADJ] A_corr {tuple(adjacency.shape)} from train only | seen-row sums "
      f"[{adjacency[seen].sum(1).min():.4f}, {adjacency[seen].sum(1).max():.4f}] "
      f"| diag min/max {adjacency.diag()[seen].min():.3f}/{adjacency.diag()[seen].max():.3f}")